# Final Project – ARTI-405 – Group 1 – Model Development
**Dylan Dyke, Eddie Kofuma-Henry, Brandon Robinson, and Caden Raleigh**

**Caden's contribution:** data cleaning, preprocessing, EDA, and feature preparation. The model-development cells below are left for Eddie and the rest of the group.

**Dataset:** [Andrei Novikov, Used Cars Dataset (Kaggle)](https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset?select=cars.csv)  
**Target:** `price` (regression). These are asking/listing prices, not verified sale prices.

> **Before running:** Download the original `cars.csv` from Kaggle and place it in `data/cars.csv` alongside this notebook (a `cars.csv` in the same folder is also supported). The dataset is **not included** in this notebook. Do not commit the large source CSV or private data to GitHub.

> **Data leakage safeguard:** This notebook cleans the records and **defines** the scikit-learn preprocessing transformer, but does **not fit** imputation, scaling, or one-hot encoding to the full dataset. Fit the transformer on **training data only** after Eddie's train/test split.

## Import Data and Packages

In [ ]:
# All package imports are in one place so we can reuse them later.
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [ ]:
# Match Dylan's notebook and keep this usable in Jupyter or VS Code.
DATA_CANDIDATES = [
    Path("data/cars.csv"),
    Path("cars.csv"),
    Path("data/raw/cars.csv"),
    Path("../data/cars.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "The Kaggle dataset is not included. Download 'cars.csv' from "
        "https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset "
        "and save it as data/cars.csv (or next to the notebook)."
    )

cars_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {DATA_PATH} | rows={len(cars_raw):,} | columns={len(cars_raw.columns)}")
print("Original column names:", cars_raw.columns.tolist())
display(cars_raw.head(5))

## Data Preprocessing

I am using only information that could reasonably be known when a person enters a car into our proposed Streamlit app. The original data remains unchanged in `cars_raw` so we can compare before/after results.

In [ ]:
# Standardize column names so later code doesn't depend on spaces or capital letters.
cars = cars_raw.copy()
cars.columns = (cars.columns.astype(str)
                .str.strip().str.lower()
                .str.replace(r"[^a-z0-9]+", "_", regex=True)
                .str.strip("_"))

# Allow a few common alternate spellings without confusing this dataset with another one.
rename_columns = {
    "make": "manufacturer",
    "milage": "mileage",
    "car_model": "model",
    "listing_price": "price",
}
cars.rename(columns={old: new for old, new in rename_columns.items()
                     if old in cars.columns and new not in cars.columns}, inplace=True)

required = {"manufacturer", "model", "year", "mileage", "transmission", "engine", "price"}
missing_required = sorted(required.difference(cars.columns))
if missing_required:
    raise ValueError(
        f"Expected the Novikov cars.csv. These required columns were missing: {missing_required}. "
        f"Found: {cars.columns.tolist()}"
    )

# Normalize category labels: e.g. " Toyota  " and "TOYOTA" become the same category.
text_columns = ["manufacturer", "model", "transmission", "engine", "fuel_type", "drivetrain",
                "exterior_color", "interior_color", "accidents_or_damage",
                "one_owner", "personal_use_only"]
for col in text_columns:
    if col in cars.columns:
        cars[col] = (cars[col].astype("string").str.strip()
                     .str.replace(r"\s+", " ", regex=True)
                     .str.lower())
        cars[col] = cars[col].replace({"": pd.NA, "nan": pd.NA,
                                      "none": pd.NA, "null": pd.NA, "n/a": pd.NA})

# Clean price, mileage, and year (numeric data may contain $ signs or commas).
def parse_number(series):
    return pd.to_numeric(
        series.astype("string").str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce",
    ).astype("float64")

for col in ["price", "mileage", "year"]:
    cars[col] = parse_number(cars[col])

# The interface sketches show "engine size". Extract the displacement (e.g. 2.0 L),
# rather than encoding long, highly variable engine descriptions as categories.
# Electric cars or descriptions without liters become missing and will be imputed later.
engine_text = cars["engine"].astype("string")
engine_liters = engine_text.str.extract(
    r"(?i)(?<!\d)(\d{1,2}(?:\.\d+)?)\s*(?:l\b|liters?\b)", expand=False
)
cars["engine_size_liters"] = pd.to_numeric(engine_liters, errors="coerce")

# Implausible numeric INPUTS are treated as missing and imputed on training data.
# Do not impute the TARGET price: rows without a legitimate target cannot be trained.
reference_year = 2023  # Kaggle source lists used-car listings from April 2023
cars.loc[~cars["year"].between(1900, reference_year + 2), "year"] = np.nan
cars.loc[~cars["mileage"].between(0, 1_000_000), "mileage"] = np.nan
cars.loc[~cars["engine_size_liters"].between(0.5, 12), "engine_size_liters"] = np.nan

original_count = len(cars)
bad_target = cars["price"].isna() | (cars["price"] <= 0)
cars = cars.loc[~bad_target].copy()
print(f"Removed {bad_target.sum():,} records with missing/nonpositive price "
      f"({original_count:,} -> {len(cars):,}).")

### Missing Values

In [ ]:
# Count missing values BEFORE any imputation; the learned fill values must come from training only.
missing_summary = pd.DataFrame({
    "missing_count": cars.isna().sum(),
    "missing_pct": (cars.isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)
print("Top missing-value columns after initial cleaning:")
display(missing_summary.head(20))

# Optional plot for the final report; these are actual counts only once cars.csv is loaded.
fig, ax = plt.subplots(figsize=(9, 5))
missing_summary.loc[missing_summary.missing_count.gt(0), "missing_pct"].head(12).sort_values().plot.barh(ax=ax)
ax.set(title="Missing Values by Column (After Initial Cleaning)", xlabel="Percent missing", ylabel="")
plt.tight_layout()
plt.show()

### Duplicate Rows

In [ ]:
# Only remove exact duplicates. Different seller listings with similar cars can still be valid.
exact_duplicates = int(cars.duplicated().sum())
cars = cars.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicate rows removed: {exact_duplicates:,}")
print(f"Records remaining: {len(cars):,}")

### Inconsistent Categories

In [ ]:
# Confirm that categorical formatting is consistent and examine the most common values.
for col in ["manufacturer", "model", "transmission"]:
    print(f"\n{col}: {cars[col].nunique(dropna=True):,} distinct values")
    display(cars[col].value_counts(dropna=False).head(10).rename("count").to_frame())

# Note: We intentionally do NOT invent a 'condition' label. The Novikov
# dataset doesn't contain a direct overall vehicle-condition rating.

### Outliers

In [ ]:
# Identify extreme values; don't blindly throw out rare expensive luxury vehicles.
# Removing every IQR outlier could bias the estimator against genuinely expensive cars.
for col in ["price", "mileage", "year", "engine_size_liters"]:
    q1, q3 = cars[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 3 * iqr, q3 + 3 * iqr
    n_flags = int(((cars[col] < low) | (cars[col] > high)).sum())
    print(f"{col}: min={cars[col].min():,.2f}, median={cars[col].median():,.2f}, "
          f"max={cars[col].max():,.2f}; 3×IQR flags={n_flags:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cars["price"].clip(upper=cars["price"].quantile(.99)).hist(bins=45, ax=axes[0])
axes[0].set(xlabel="Price ($), top 1% clipped for plotting", ylabel="Listings", title="Used Car Listing Prices")
cars["mileage"].clip(upper=cars["mileage"].quantile(.99)).hist(bins=45, ax=axes[1])
axes[1].set(xlabel="Mileage, top 1% clipped for plotting", ylabel="Listings", title="Mileage Distribution")
plt.tight_layout()
plt.show()

## Exploratory Data Analysis

In [ ]:
# Quick exploratory relationships useful for the presentation and report.
print("Price summary (asking prices in USD):")
display(cars["price"].describe(percentiles=[.10, .25, .50, .75, .90, .99]).to_frame("price"))

# Make/year and price: aggregate to avoid plotting hundreds of thousands of points.
by_year = cars.groupby("year", dropna=True)["price"].median().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
by_year.plot(ax=ax)
ax.set(title="Median Listing Price by Model Year", xlabel="Model Year", ylabel="Median Asking Price ($)")
plt.tight_layout()
plt.show()

print("Spearman correlations among numeric variables:")
display(cars[["price", "year", "mileage", "engine_size_liters"]].corr(method="spearman").round(3))

### Feature Selection

In [ ]:
# Select columns that the Streamlit user can realistically enter.
# These align with the group's draft UI (manufacturer, model, mileage,
# transmission, engine size) plus model year, a major price factor.
# We leave out price_drop (post-listing information), seller_name, seller_rating,
# driver reviews, and other attributes not in the proposed UI.
NUMERIC_FEATURES = ["year", "mileage", "engine_size_liters"]
CATEGORICAL_FEATURES = ["manufacturer", "model", "transmission"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "price"

X = cars[FEATURE_COLUMNS].copy()
# scikit-learn imputers expect np.nan, not pandas <NA>, with object/string columns.
# Converting here avoids ambiguous-NA errors during Eddie's model training.
for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
for col in NUMERIC_FEATURES:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype("float64")
y = cars[TARGET_COLUMN].copy()
print(f"Features: {FEATURE_COLUMNS}")
print(f"X shape: {X.shape}; y shape: {y.shape}")
print("Missing values in selected features (handled by the pipeline):")
display(X.isna().sum().rename("missing_count").to_frame())

### Normalization

The team wants model inputs scaled between **0 and 1**, so numeric columns use `MinMaxScaler`. Categorical inputs use one-hot encoding (0/1 indicator columns). **The price target remains in dollars**. The transformer must be fitted only after the data has been split into training and testing sets.

#### Scaling

In [ ]:
# Numeric pipeline: learn the median from training rows only, then scale to [0,1]
# based on the training range. Unseen test data can fall just outside 0-1;
# clip=True keeps it in [0,1] for deployment.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler(clip=True)),
])
print(numeric_pipeline)

#### One-Hot Encoding

In [ ]:
# Min frequency combines infrequent model names; useful with thousands of car models.
# Unknown categories during deployment are ignored instead of crashing the app.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=25)),
])

def make_preprocessor():
    """Create a fresh, UNFITTED transformer for training. Never fit to all X."""
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

preprocessor = make_preprocessor()
print(preprocessor)

### Display Prepared Data

In [ ]:
# Caden's cleaned feature matrix and targets are ready to be split by the modeling team.
# These are RAW selected values: the preprocessor is intentionally UNFITTED.
display(X.head(5))
print(f"Cleaned labeled records: {len(X):,}")
print("Numeric fields to be imputed/scaled:", NUMERIC_FEATURES)
print("Categorical fields to be imputed/encoded:", CATEGORICAL_FEATURES)
print("PREPROCESSOR FITTED?", hasattr(preprocessor, "transformers_"))

### Handoff to Eddie — how to use the cleaned data without leakage

The following cell is **illustrative** and intentionally not executed in this section. The model developer can place it under *Split Data* below. All models should use the same held-out test set. The MLP may require converting the sparse output to a dense array after fitting, subject to memory constraints.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
preprocessor = make_preprocessor()
X_train_ready = preprocessor.fit_transform(X_train)  # fit TRAIN ONLY
X_test_ready = preprocessor.transform(X_test)       # never fit on test

# Save a fitted preprocessing + prediction pipeline together, so the
# Streamlit app applies the exact same feature transformations.
# example: full_pipeline = Pipeline([('preprocessor', make_preprocessor()),
#                                    ('model', RandomForestRegressor(...))])
#          full_pipeline.fit(X_train, y_train)
```

**UI note:** The dataset has no direct overall *condition* field. If the final app asks for condition, we need either a different source or an explicitly labeled proxy, not invented training data. The model should be evaluated for errors across makes, years, and mileage bands. For transparency, the output is an **estimated listing price**, not a guaranteed fair or sale price.

**Important:** We didn't fit an encoder before the split, didn't train a model, and didn't fabricate accuracy values or missing-value statistics.

## Model Development

### Class Imbalance

*Not applicable to this regression target:* class imbalance is a classification concern. For price prediction, examine skewed price ranges instead. **Eddie's model-development section starts here.**

### Split Data

*Assigned to Eddie / model development — use the handoff example above.*

### Multiple Linear Regression (Baseline)

*Tuning description and justification — assigned to Eddie / model development.*

### Multilayered Perceptron

*Tuning description and justification — assigned to Eddie / model development.*

### Random Forest

*Tuning description and justification — assigned to Eddie / model development.*

### Model 4

*Tuning description and justification — assigned to Eddie / model development.*

### Model 5

*Tuning description and justification — assigned to Eddie / model development.*

## Model Comparison

### Comparison Table

*To be completed after the models are trained and evaluated on the same held-out test set.*

### Reasoning

*Explain the model selection using R², MAE/RMSE, generalization, and runtime.*

## Final Selection

*To be completed after model evaluation.*